In [1]:

#Imports
import sys
import os
sys.path.append(os.path.abspath(".."))


from src.data import load_matches, load_elo_baseline
from src.elo import prepare_matches, run_elo_updates, update_elo
from src.features import rolling_form, h2h_last_five, average_goal_diff, build_prediction_features

import pandas as pd
from collections import defaultdict

sys.path.append(os.path.abspath(".."))



In [7]:
# Load data
df = load_matches() #Load dataframe containing all historical matches
country_elo = load_elo_baseline() #Generate a dictionary with every country having their elo from 12/13/2025

# Prepare and update
recent_matches = prepare_matches(df, start_date="2025-12-13", end_date="2026-06-11") #We need these games since the current elo is only accurate up til 12/13/2025
country_elo = run_elo_updates(recent_matches, country_elo)

#Top 20
top20 = sorted(country_elo.items(), key=lambda x: x[1], reverse=True)[:20]
print("Top 20 teams going into the WC based off ELO")
for rank, (team, rating) in enumerate(top20, 1):
    print(f"{rank}. {team}: {rating:.0f}")

Top 20 teams going into the WC based off ELO
1. Spain: 2163
2. Argentina: 2113
3. France: 2081
4. England: 2020
5. Brazil: 1990
6. Portugal: 1984
7. Colombia: 1975
8. Netherlands: 1962
9. Ecuador: 1937
10. Croatia: 1931
11. Germany: 1926
12. Norway: 1912
13. Japan: 1906
14. Turkey: 1902
15. Switzerland: 1893
16. Uruguay: 1892
17. Morocco: 1876
18. Mexico: 1868
19. Belgium: 1866
20. Denmark: 1859


ELO caculations turn out essentially the same to the elo rating found on https://www.eloratings.net/2026_World_Cup, 

In [4]:
current_elos = defaultdict(lambda: 1500)
training_rows = []
WC_START = pd.Timestamp("2026-06-11")

wc_teams = ["Canada", "Mexico", "United States", "Panama", "Costa Rica", "Jamaica", "Argentina", "Brazil", "Uruguay", "Colombia", "Ecuador", "Paraguay", "Venezuela", "Spain", "France", "England", "Portugal", "Netherlands", "Belgium", "Italy", "Germany", "Croatia", "Switzerland", "Denmark", "Ukraine", "Turkey", "Austria", "Norway", "Serbia", "Morocco", "Senegal", "Nigeria", "Egypt", "Ivory Coast", "Tunisia", "Algeria", "Mali", "Japan", "South Korea", "Iran", "Australia", "Saudi Arabia", "Iraq", "Qatar", "Uzbekistan", "New Zealand"]


df = prepare_matches(df, df['date'].min(), WC_START)

# Sort matches chronologically
df_sorted = df.sort_values('date')

for i, row in df_sorted.iterrows():

    home_elo = current_elos[row['home_team']]
    away_elo = current_elos[row['away_team']]
    home_h2h, away_h2h = h2h_last_five(df_sorted, row['home_team'], row['away_team'])
    # We pass the current row's date so the functions don't look into the future
    match_features = {
        'date': row['date'],
        'home_team': row['home_team'],
        'away_team': row['away_team'],
        'elo_diff': home_elo - away_elo,
        'home_form': rolling_form(df_sorted, row['home_team'], row['date']),
        'away_form': rolling_form(df_sorted, row['away_team'], row['date']),
        'home_h2h': home_h2h,
        'away_h2h': away_h2h,
        'neutral': row['neutral'],
        'target': row['result'] # 1, 0.5, or 0
    }
    training_rows.append(match_features)
    
    new_away, new_home = update_elo(
        row['result'], row['neutral'], row['K_factor'], 
        row['home_score'], row['away_score'], 
        away_elo, home_elo
    )
    current_elos[row['home_team']] = new_home
    current_elos[row['away_team']] = new_away

# 3. Finalize
train_df = pd.DataFrame(training_rows)

In [6]:
train_df.to_csv("/Users/armand_k/Workout app/WC 2026/data/world_cup_training_data.csv", index=False)

print("File saved successfully!")

File saved successfully!
